# Finale · one question, four versions (slides 68–69)

"How did Acme's Q3 margin compare to Q2, and why?" through every version we built:

| Version | What it is |
|---|---|
| v0 | the whole filing in every prompt (`v0_whole_doc.ipynb`) |
| v1 | basic RAG, top 5 chunks (`v1_basic_rag.ipynb`) |
| v2 | a LangGraph agent with `search_docs` (`v2_langgraph_agent.ipynb`) |
| v3 | agentic RAG, served on AgentShip (`v3_agentic_rag.ipynb`, `agents/agentic_rag.yaml`) |

Each version runs `RUNS` times, so p95 latency means something. With a self-hosted model, v0's
first run reads the whole ~100K-token filing before it writes a word; run this notebook once
before the lecture to warm the server's cache.

**Setup:** see `README.md` (`pip install -e .`, then `.env` names the model and server). Run with the *Agentic RAG demos* kernel. v3 needs Postgres for its checkpoints: `docker compose up -d`.

In [ ]:
import os

os.environ.setdefault("AGENTSHIP_OTEL_EXPORTERS", "")  # set to phoenix to see v3's traces at :6006

import numpy as np
import pandas as pd

import eval as E  # the same version functions eval.py scores

QUESTION = "How did Acme's Q3 margin compare to Q2, and why?"
VERSIONS = ["v0", "v1", "v2", "v3"]
RUNS = 3

In [ ]:
rows, answers = [], {}
for version in VERSIONS:
    for i in range(RUNS):
        run = E.VERSIONS[version](QUESTION)
        rows.append({"version": version, "input_tok": run.prompt_tokens, "searches": len(run.searches),
                     "cost": run.cost, "latency": run.latency})
        print(f"{version} run {i + 1}: {run.prompt_tokens:,} input tokens, {len(run.searches)} searches, "
              f"{run.latency:.1f}s")
    answers[version] = run

## The answers

In [ ]:
for version, run in answers.items():
    searched = f"  searched: {run.searches}" if run.searches else ""
    print(f"── {version} ──{searched}\n{run.answer}\n")

## The numbers, side by side (slide 69)

In [ ]:
df = pd.DataFrame(rows)
table = df.groupby("version", sort=False).agg(
    input_tok=("input_tok", "median"), searches=("searches", "median"),
    usd_per_query=("cost", "mean"), p95_s=("latency", lambda s: np.percentile(s, 95)))
table["input_tok"] = table["input_tok"].map(lambda n: f"{n / 1000:.1f}K")
table["usd_per_query"] = table["usd_per_query"].map(lambda c: f"${c:.3f}" if pd.notna(c) else "n/a")
table["p95_s"] = table["p95_s"].map(lambda s: f"{s:.1f}s")
table

Same question, four versions. v3, served on AgentShip, is what ships.

$ is priced at `PRICE_AS` when the model has no list price (a self-hosted model). For the whole
golden set, with retrieval and answer quality scored too: `python eval.py --agents v0,v1,v2,v3`.